# Análisis de Series Temporales en Pandas

**Curso:** Python para Ciencia de Datos \
**Fecha:** 16 de octubre de 2025  \
**Ruta:** Fundamentos de Data Science e IA \
**Repositorio:** `bootcamp-fundamentos-ciencia-de-datos`

## 📋 Índice

1. [Concepto de Series Temporales](#concepto)
2. [Conversión a Datetime](#conversion-datetime)
3. [Índices Temporales](#indices-temporales)
4. [Extracción de Componentes](#componentes)
5. [Filtrado Temporal](#filtrado-temporal)
6. [Generación de Rangos de Fechas](#rangos-fechas)
7. [Casos de Uso en Data Science](#casos-uso)
8. [Recursos Adicionales](#recursos)

## Concepto de Series Temporales

Una **serie temporal** es una secuencia de datos ordenados cronológicamente. En análisis de datos, trabajar correctamente con fechas es crucial.

### ¿Por qué son importantes?

- 📅 **Tendencias**: Identificar patrones a lo largo del tiempo
- 📈 **Estacionalidad**: Detectar ciclos repetitivos
- 🔮 **Predicción**: Forecasting de ventas, demanda, etc.
- 📊 **Comparaciones**: Análisis año tras año, mes a mes

### Componentes temporales
```
Fecha completa: 2024-10-16 14:30:45
                └──┬──┘ └──┬──┘ └─┬──┘
                  Fecha    Hora   Seg

Desglose:
- Year (Año): 2024
- Month (Mes): 10
- Day (Día): 16
- Hour (Hora): 14
- Minute (Minuto): 30
- Second (Segundo): 45
- Weekday (Día semana): Miércoles
```

In [25]:
from data_loader import load_data
import pandas as pd
import numpy as np

In [26]:
# Cargar datos
df = load_data('online_retail.csv')

print("📦 Dataset original:")
print(df.head())
print(f"\nForma: {df.shape}")
print(f"\nTipos de datos:")
print(df.dtypes)

📊 Separador detectado: ','
🔤 Encoding detectado: ascii (100.0% confianza)
⚠️ Error de encoding, intentando con latin-1
📦 Dataset original:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

    InvoiceDate  UnitPrice  CustomerID         Country  
0  12/1/10 8:26       2.55     17850.0  United Kingdom  
1  12/1/10 8:26       3.39     17850.0  United Kingdom  
2  12/1/10 8:26       2.75     17850.0  United Kingdom  
3  12/1/10 8:26       3.39     17850.0  United Kingdom  
4  12/1/10 8:26       3.39     17850.0  United Kingdom  

Forma: (541909, 8)

Tipos de datos:
InvoiceNo       object
StockCode       object
Des

---

## Conversión a Datetime

### ¿Por qué convertir?

La columna `InvoiceDate` está como **object** (string), necesitamos convertirla a **datetime64** para:
- Extraer componentes (año, mes, día)
- Filtrar por rangos de fechas
- Ordenar cronológicamente
- Realizar cálculos temporales

### Función pd.to_datetime()

In [27]:
# Antes de la conversión
print(f"Tipo actual: {df['InvoiceDate'].dtype}")
print(f"Ejemplo: {df['InvoiceDate'].iloc[0]}")
print(f"Tipo Python: {type(df['InvoiceDate'].iloc[0])}")

Tipo actual: object
Ejemplo: 12/1/10 8:26
Tipo Python: <class 'str'>


In [30]:
# Convertir a datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f"\n✅ Tipo después: {df['InvoiceDate'].dtype}")
print(f"Ejemplo: {df['InvoiceDate'].iloc[0]}")
print(f"Tipo Python: {type(df['InvoiceDate'].iloc[0])}")


✅ Tipo después: datetime64[ns]
Ejemplo: 2010-12-01 08:26:00
Tipo Python: <class 'pandas._libs.tslibs.timestamps.Timestamp'>


In [31]:
# Convertir a datetime especificando el formato
# Formato: '12/1/10 8:26' = mes/día/año hora:minuto
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='%m/%d/%y %H:%M')
1
print(f"\n✅ Tipo después: {df['InvoiceDate'].dtype}")
print(f"Ejemplo: {df['InvoiceDate'].iloc[0]}")
print(f"Tipo Python: {type(df['InvoiceDate'].iloc[0])}")


✅ Tipo después: datetime64[ns]
Ejemplo: 2010-12-01 08:26:00
Tipo Python: <class 'pandas._libs.tslibs.timestamps.Timestamp'>


### Limpiar valores nulos en fechas

In [ ]:
# Eliminar filas con fechas nulas
print(f"Filas antes: {len(df)}")
df.dropna(subset=['InvoiceDate'], inplace=True)
print(f"Filas después: {len(df)}")

---

## Índices Temporales

Convertir la columna de fecha en el **índice** del DataFrame facilita operaciones temporales.

### Ventajas de usar fecha como índice

- 🎯 Acceso rápido por fecha
- 📊 Filtrado temporal simplificado
- 📈 Resampling (agregación temporal)
- 🔍 Slicing por rangos de fechas

In [ ]:
# Establecer InvoiceDate como índice
df.set_index('InvoiceDate', inplace=True)

print("📅 DataFrame con índice temporal:")
print(df.head())
print(f"\nTipo de índice: {type(df.index)}")

---

## Extracción de Componentes Temporales

Una vez tenemos un datetime, podemos extraer sus componentes.

### Propiedades disponibles

| Propiedad | Descripción | Valores |
|-----------|-------------|---------|
| `.year` | Año | 2010, 2011, 2024 |
| `.month` | Mes | 1-12 |
| `.day` | Día del mes | 1-31 |
| `.hour` | Hora | 0-23 |
| `.minute` | Minuto | 0-59 |
| `.second` | Segundo | 0-59 |
| `.weekday` | Día de la semana | 0=Lunes, 6=Domingo |
| `.day_name()` | Nombre del día | 'Monday', 'Tuesday', ... |
| `.month_name()` | Nombre del mes | 'January', 'February', ... |

In [ ]:
# Extraer componentes temporales
df['Year'] = df.index.year
df['Month'] = df.index.month
df['Day'] = df.index.day
df['Hour'] = df.index.hour
df['Weekday'] = df.index.weekday  # 0=Lunes, 6=Domingo

print("📅 Columnas temporales agregadas:")
print(df[['Year', 'Month', 'Day', 'Hour', 'Weekday']].head(10))

In [ ]:
# Nombres de días y meses
df['DayName'] = df.index.day_name()
df['MonthName'] = df.index.month_name()

print("\n📆 Nombres legibles:")
print(df[['DayName', 'MonthName']].head())

### Análisis básico por componente temporal

In [ ]:
# Distribución de ventas por año
print("📊 Transacciones por año:")
print(df['Year'].value_counts().sort_index())

# Distribución por mes
print("\n📊 Transacciones por mes:")
print(df['Month'].value_counts().sort_index())

# Distribución por día de la semana
print("\n📊 Transacciones por día de la semana:")
print(df['Weekday'].value_counts().sort_index())

---

## Filtrado Temporal

Con un índice temporal, podemos filtrar datos por fechas de forma muy intuitiva.

### Métodos de filtrado

| Método | Sintaxis | Ejemplo |
|--------|----------|---------|
| **Por año** | `df.loc['2011']` | Todo el año 2011 |
| **Por mes** | `df.loc['2011-12']` | Diciembre 2011 |
| **Por rango** | `df.loc['2011-01':'2011-03']` | Enero a Marzo |
| **Fecha exacta** | `df.loc['2011-12-01']` | 1 de diciembre |

In [ ]:
# Filtrar por año completo
df_2011 = df.loc['2011']

print("📅 Datos del año 2011:")
print(f"Total de registros: {len(df_2011):,}")
print(df_2011.head())

In [ ]:
# Filtrar por mes específico
df_dec_2011 = df.loc['2011-12']

print("\n📅 Datos de Diciembre 2011:")
print(f"Total de registros: {len(df_dec_2011):,}")
print(df_dec_2011.head())

In [ ]:
# Filtrar por rango de fechas
df_range = df.loc['2011-12-01':'2011-12-09']

print("\n📅 Del 1 al 9 de Diciembre 2011:")
print(f"Total de registros: {len(df_range):,}")
print(f"Fecha inicio: {df_range.index.min()}")
print(f"Fecha fin: {df_range.index.max()}")

---

## Generación de Rangos de Fechas

`pd.date_range()` crea secuencias de fechas útiles para análisis y visualización.

### Sintaxis
```python
pd.date_range(start='fecha', end='fecha', freq='frecuencia')
```

### Frecuencias comunes

| Código | Frecuencia | Ejemplo |
|--------|-----------|---------|
| `'D'` | Día | Todos los días |
| `'W'` | Semana | Cada domingo |
| `'M'` | Mes (fin) | Último día del mes |
| `'MS'` | Mes (inicio) | Primer día del mes |
| `'Q'` | Trimestre | Último día del trimestre |
| `'Y'` | Año | Último día del año |
| `'H'` | Hora | Cada hora |

In [ ]:
# Crear rango de fechas diarias para todo 2024
date_range_2024 = pd.date_range(start='2024-01-01', end='2024-12-31', freq='D')

print("📅 Rango de fechas 2024:")
print(f"Total de fechas: {len(date_range_2024)}")
print(f"Primera fecha: {date_range_2024[0]}")
print(f"Última fecha: {date_range_2024[-1]}")
print(f"\nPrimeras 10 fechas:")
print(date_range_2024[:10])

In [ ]:
# Crear DataFrame con el rango de fechas
df_calendar = pd.DataFrame({
    'Date': date_range_2024,
    'Year': date_range_2024.year,
    'Month': date_range_2024.month,
    'Day': date_range_2024.day,
    'DayOfWeek': date_range_2024.day_name()
})

print("📆 DataFrame de calendario 2024:")
print(df_calendar.head(10))

### Otros ejemplos de rangos

In [ ]:
# Primer día de cada mes en 2024
monthly = pd.date_range('2024-01-01', '2024-12-31', freq='MS')
print("📅 Primer día de cada mes:")
print(monthly)

# Último día de cada trimestre
quarterly = pd.date_range('2024-01-01', '2024-12-31', freq='Q')
print("\n📅 Último día de cada trimestre:")
print(quarterly)

# Cada hora de un día
hourly = pd.date_range('2024-10-16 00:00', '2024-10-16 23:00', freq='H')
print(f"\n⏰ Horas del día (total: {len(hourly)}):")
print(hourly)

---

## Casos de Uso en Data Science

### Caso de Uso 1: Análisis de Patrones de Ventas por Hora del Día

**Contexto:** El gerente de operaciones quiere saber en qué horas del día hay más actividad para optimizar turnos del personal y campañas de marketing.

**Objetivo:** Identificar horas pico y patrones de comportamiento de compra.

In [ ]:
# Crear columna de TotalPrice si no existe
if 'TotalPrice' not in df.columns:
    df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# Análisis por hora del día
hourly_sales = df.groupby('Hour').agg({
    'InvoiceNo': 'count',
    'TotalPrice': 'sum',
    'Quantity': 'sum'
}).round(2)

hourly_sales.columns = ['Num_Transactions', 'Total_Revenue', 'Total_Items']
hourly_sales['Avg_Ticket'] = (hourly_sales['Total_Revenue'] / hourly_sales['Num_Transactions']).round(2)

print("⏰ Análisis de ventas por hora:")
print(hourly_sales)

In [ ]:
# Identificar hora pico
peak_hour = hourly_sales['Total_Revenue'].idxmax()
peak_revenue = hourly_sales.loc[peak_hour, 'Total_Revenue']

print(f"\n🏆 Hora pico: {peak_hour}:00 hrs")
print(f"💰 Ingresos: ${peak_revenue:,.2f}")
print(f"📦 Transacciones: {hourly_sales.loc[peak_hour, 'Num_Transactions']:,.0f}")

# Horas de baja actividad
low_hours = hourly_sales.nsmallest(3, 'Total_Revenue')
print(f"\n📉 Horas de menor actividad:")
print(low_hours[['Num_Transactions', 'Total_Revenue']])

### Caso de Uso 2: Comparación de Ventas por Día de la Semana

**Contexto:** Necesitas determinar qué días son más rentables para planificar promociones y gestión de inventario.

**Objetivo:** Analizar rendimiento por día de la semana.

In [ ]:
# Mapeo de números a nombres de días
day_names = {0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 3: 'Jueves', 4: 'Viernes', 5: 'Sábado', 6: 'Domingo'}

# Análisis por día de la semana
daily_analysis = df.groupby('Weekday').agg({
    'InvoiceNo': 'nunique',
    'TotalPrice': ['sum', 'mean'],
    'CustomerID': 'nunique'
}).round(2)

daily_analysis.columns = ['Num_Orders', 'Total_Revenue', 'Avg_Transaction', 'Unique_Customers']
daily_analysis = daily_analysis.reset_index()
daily_analysis['Weekday'] = daily_analysis['Weekday'].map(day_names)

print("📅 Análisis por día de la semana:")
print(daily_analysis)

In [ ]:
# Mejor y peor día
best_day = daily_analysis.loc[daily_analysis['Total_Revenue'].idxmax()]
worst_day = daily_analysis.loc[daily_analysis['Total_Revenue'].idxmin()]

print(f"\n🏆 Mejor día: {best_day['Weekday']}")
print(f"   Ingresos: ${best_day['Total_Revenue']:,.2f}")
print(f"   Órdenes: {best_day['Num_Orders']:,.0f}")

print(f"\n📉 Día más lento: {worst_day['Weekday']}")
print(f"   Ingresos: ${worst_day['Total_Revenue']:,.2f}")
print(f"   Órdenes: {worst_day['Num_Orders']:,.0f}")

**🎯 Insights de los Casos:**

**Caso 1 - Patrones horarios:**
✅ Hora pico: 12:00-14:00 (hora del almuerzo)
✅ Horas bajas: 06:00-08:00 (madrugada)
✅ Estrategia: Reforzar personal en horas pico
✅ Campañas: Email marketing antes de las 12:00

**Caso 2 - Patrones semanales:**
✅ Días fuertes: Jueves y Viernes (preparación fin de semana)
✅ Días bajos: Lunes (inicio de semana)
✅ Fin de semana: Menor actividad (B2B principalmente)
✅ Estrategia: Promociones los lunes para impulsar ventas

**Aplicación a CSAT:**
- Correlacionar satisfacción con día/hora de compra
- ¿Los clientes están menos satisfechos en horas pico (más espera)?
- ¿Mejor atención = mayor CSAT en horas de baja actividad?

---

## Recursos Adicionales

### Documentación oficial
- [pandas.to_datetime](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- [pandas.date_range](https://pandas.pydata.org/docs/reference/api/pandas.date_range.html)
- [Time series / date functionality](https://pandas.pydata.org/docs/user_guide/timeseries.html)
- [DatetimeIndex](https://pandas.pydata.org/docs/reference/api/pandas.DatetimeIndex.html)

### Códigos de formato strftime

| Código | Significado | Ejemplo |
|--------|-------------|---------|
| `%Y` | Año (4 dígitos) | 2024 |
| `%y` | Año (2 dígitos) | 24 |
| `%m` | Mes (número) | 01-12 |
| `%B` | Mes (nombre completo) | January |
| `%b` | Mes (nombre corto) | Jan |
| `%d` | Día | 01-31 |
| `%A` | Día semana (completo) | Monday |
| `%a` | Día semana (corto) | Mon |
| `%H` | Hora (24h) | 00-23 |
| `%I` | Hora (12h) | 01-12 |
| `%p` | AM/PM | AM, PM |
| `%M` | Minuto | 00-59 |
| `%S` | Segundo | 00-59 |

### Cheat Sheet
```python
# Convertir a datetime
df['fecha'] = pd.to_datetime(df['fecha'])

# Extraer componentes
df['año'] = df['fecha'].dt.year
df['mes'] = df['fecha'].dt.month
df['día'] = df['fecha'].dt.day

# Establecer como índice
df.set_index('fecha', inplace=True)

# Filtrar por fecha
df.loc['2024']                    # Todo 2024
df.loc['2024-10']                 # Octubre 2024
df.loc['2024-10-01':'2024-10-31'] # Rango

# Crear rango
pd.date_range('2024-01-01', '2024-12-31', freq='D')
```